# 11. InfraScan Crack Segmentation Test Evaluation

This notebook evaluates the winning crack segmentation model trained on Kaggle against:
1. **Crack500 In-Domain Test Split** (1,087 images)
2. **CFD Cross-Domain External Test Set** (118 images)

> **Scientific Integrity & Anti-Snooping Guard**:
> - Model architecture, checkpoint epoch, and threshold were selected **strictly on the Validation split**.
> - Test splits are evaluated **exactly once** with single-run guard protection (`FORCE_RERUN=False`).
> - Pre-evaluation validation reproducibility check is verified prior to running test splits.


### 1. Environment Configuration and Reproducibility Setup
Initialize deterministic seed (42), set path configurations, and select computing device.

In [1]:
import os
import sys
from pathlib import Path
import random
import numpy as np
import torch
import pandas as pd

# Deterministic seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Configure project and data paths
PROJECT_ROOT = Path(r"d:\Infosys springboards\infrascan")
DATA_ROOT = Path(r"d:\Infosys springboards\data")
MODELS_DIR = PROJECT_ROOT / "ml" / "models" / "segmentation"
MANIFESTS_DIR = DATA_ROOT / "processed" / "manifests"

sys.path.insert(0, str(PROJECT_ROOT))
from ml.segmentation import evaluate_seg_test as est

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[SETUP] Execution Device: {DEVICE}")
print(f"[SETUP] Models Directory: {MODELS_DIR} (exists: {MODELS_DIR.exists()})")
print(f"[SETUP] Manifests Directory: {MANIFESTS_DIR} (exists: {MANIFESTS_DIR.exists()})")


[SETUP] Execution Device: cpu
[SETUP] Models Directory: d:\Infosys springboards\infrascan\ml\models\segmentation (exists: True)
[SETUP] Manifests Directory: d:\Infosys springboards\data\processed\manifests (exists: True)


### 2. Inspect Model Checkpoints & Select Winning Architecture by Validation Dice
Inspect run cards in `ml/models/segmentation/` and select the winner based strictly on validation split Dice.

In [2]:
# Inspect candidate models and choose winner strictly by validation Dice
winner_info = est.get_winner_run(MODELS_DIR)

print("=" * 80)
print(f"WINNING MODEL SELECTED: {winner_info['run_name']} ({winner_info['model_name'].upper()})")
print(f"  Encoder Backbone:      {winner_info['encoder']} (Pretrained weights: {winner_info['encoder_weights']})")
print(f"  Optimal Val Threshold: {winner_info['optimal_threshold']:.2f}")
print(f"  Best Validation Dice:  {winner_info['val_dice']:.4f}")
print("=" * 80)

print("\n--- All Trained Segmentation Candidates ---")
candidates_df = pd.DataFrame([
    {
        "Run Name": c["run_name"],
        "Architecture": c["model_name"],
        "Encoder": c["encoder"],
        "Encoder Weights": c["encoder_weights"],
        "Optimal Threshold": c["optimal_threshold"],
        "Validation Dice": c["val_dice"],
        "Selection Status": "WINNER" if c["run_name"] == winner_info["run_name"] else "Runner-up"
    }
    for c in winner_info["all_candidates"]
])
print(candidates_df.to_string(index=False))

# Load winner model weights
model = est.load_model_from_checkpoint(winner_info, DEVICE)
EVAL_DIR = winner_info["run_dir"] / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n[OK] Model successfully loaded from: {winner_info['checkpoint_path']}")
print(f"[OK] Evaluation artifacts destination: {EVAL_DIR}")


WINNING MODEL SELECTED: unetplusplus_crack500 (UNETPLUSPLUS)
  Encoder Backbone:      resnet34 (Pretrained weights: imagenet)
  Optimal Val Threshold: 0.40
  Best Validation Dice:  0.7482

--- All Trained Segmentation Candidates ---
             Run Name Architecture  Encoder Encoder Weights  Optimal Threshold  Validation Dice Selection Status
unetplusplus_crack500 unetplusplus resnet34        imagenet               0.40           0.7482           WINNER
        unet_crack500         unet resnet34        imagenet               0.25           0.7335        Runner-up

[OK] Model successfully loaded from: d:\Infosys springboards\infrascan\ml\models\segmentation\unetplusplus_crack500\best.pth
[OK] Evaluation artifacts destination: d:\Infosys springboards\infrascan\ml\models\segmentation\unetplusplus_crack500\evaluation


### 3. Pre-Evaluation Validation Reproducibility Check
Re-evaluate the Crack500 validation split to verify that preprocessing and weights match training exactly (|Diff| <= 0.01).

In [3]:
# Validate that local evaluation pipeline reproduces training validation metrics
val_manifest = MANIFESTS_DIR / "crack500.csv"
val_results = est.verify_reproducibility(
    model=model,
    winner_info=winner_info,
    val_manifest_csv=val_manifest,
    device=DEVICE,
    tolerance=0.01
)

print("=" * 80)
print("[REPRODUCIBILITY CHECK PASSED]")
print(f"  Validation Samples:       {val_results['val_samples']}")
print(f"  Operating Threshold:      {val_results['threshold']:.2f}")
print(f"  Expected Val Dice (Card): {val_results['expected_val_dice']:.4f}")
print(f"  Computed Val Dice (Local):{val_results['val_dice']:.4f}")
print(f"  Absolute Difference:      {val_results['dice_diff']:.6f} (tolerance: 0.01)")
print(f"  Computed Val IoU:         {val_results['val_iou']:.4f}")
print(f"  Computed Val Precision:   {val_results['val_precision']:.4f}")
print(f"  Computed Val Recall:      {val_results['val_recall']:.4f}")
print("=" * 80)


[REPRODUCIBILITY CHECK] Processed 50/341 validation images...
[REPRODUCIBILITY CHECK] Processed 100/341 validation images...
[REPRODUCIBILITY CHECK] Processed 150/341 validation images...
[REPRODUCIBILITY CHECK] Processed 200/341 validation images...
[REPRODUCIBILITY CHECK] Processed 250/341 validation images...
[REPRODUCIBILITY CHECK] Processed 300/341 validation images...
[REPRODUCIBILITY CHECK] Processed 341/341 validation images...
[REPRODUCIBILITY CHECK PASSED]
  Validation Samples:       341
  Operating Threshold:      0.40
  Expected Val Dice (Card): 0.7482
  Computed Val Dice (Local):0.7482
  Absolute Difference:      0.000005 (tolerance: 0.01)
  Computed Val IoU:         0.6265
  Computed Val Precision:   0.7698
  Computed Val Recall:      0.7962


### 4. Single-Pass In-Domain Test Evaluation (Crack500 Test Split)
Evaluate the held-out Crack500 test set (1,087 images) exactly once. Saves masks, probability maps, and metrics.

In [4]:
# Single-pass evaluation on Crack500 test split with single-run test guard
crack500_manifest = MANIFESTS_DIR / "crack500.csv"
FORCE_RERUN = False  # Set to True only if explicitly overwriting test split results

c500_metrics, c500_per_image_df = est.evaluate_dataset_split(
    model=model,
    split_name="crack500",
    manifest_csv=crack500_manifest,
    threshold=winner_info["optimal_threshold"],
    eval_dir=EVAL_DIR,
    device=DEVICE,
    model_name=winner_info["model_name"],
    force_rerun=FORCE_RERUN,
    max_prob_maps=50
)

print("\n--- Crack500 Test Split Metrics Summary ---")
for k, v in c500_metrics.items():
    print(f"  {k}: {v}")


[CRACK500] Starting single-pass evaluation on 1087 images (threshold=0.40)...
[CRACK500] Processed 50/1087 images...
[CRACK500] Processed 100/1087 images...
[CRACK500] Processed 150/1087 images...
[CRACK500] Processed 200/1087 images...
[CRACK500] Processed 250/1087 images...
[CRACK500] Processed 300/1087 images...
[CRACK500] Processed 350/1087 images...
[CRACK500] Processed 400/1087 images...
[CRACK500] Processed 450/1087 images...
[CRACK500] Processed 500/1087 images...
[CRACK500] Processed 550/1087 images...
[CRACK500] Processed 600/1087 images...
[CRACK500] Processed 650/1087 images...
[CRACK500] Processed 700/1087 images...
[CRACK500] Processed 750/1087 images...
[CRACK500] Processed 800/1087 images...
[CRACK500] Processed 850/1087 images...
[CRACK500] Processed 900/1087 images...
[CRACK500] Processed 950/1087 images...
[CRACK500] Processed 1000/1087 images...
[CRACK500] Processed 1050/1087 images...
[CRACK500] Processed 1087/1087 images...
[CRACK500] Evaluation complete. Metrics:

### 5. Crack500 Data Leakage & Near-Duplicate Check
Compute 64-bit dHash (9x8 grayscale) and parent image ID overlap between train and test splits.

In [5]:
# Crack500 train-to-test leakage check via 64-bit dHash and parent crop ID analysis
leakage_report = est.check_crack500_leakage(
    train_manifest_csv=crack500_manifest,
    test_manifest_csv=crack500_manifest,
    test_per_image_df=c500_per_image_df,
    eval_dir=EVAL_DIR
)

print("=" * 80)
print("[CRACK500 LEAKAGE CHECK REPORT]")
print(f"  Train Image Count:           {leakage_report['train_image_count']}")
print(f"  Test Image Count:            {leakage_report['test_image_count']}")
print(f"  Shared Parent Images:        {leakage_report['shared_parents_count']} (Leakage: {leakage_report['parent_leakage_detected']})")
print(f"  Hamming Dist 0 (Duplicates): {leakage_report['hamming_dist_0_exact_duplicates']}")
print(f"  Hamming Dist 1-2 (Near Dups):{leakage_report['hamming_dist_1_near_duplicates'] + leakage_report['hamming_dist_2_near_duplicates']}")
print(f"  Mean Min Hamming Distance:   {leakage_report['mean_min_hamming_distance']:.2f} bits")
print(f"  Dice (With Near Duplicate):  {leakage_report['mean_dice_with_near_duplicate']:.4f}")
print(f"  Dice (Without Duplicate):    {leakage_report['mean_dice_without_near_duplicate']:.4f}")
print(f"  Dice Difference:             {leakage_report['dice_difference']:.4f}")
print("=" * 80)


Computing 64-bit dHash for Crack500 train images (1860)...
Computing 64-bit dHash for Crack500 test images (1087)...


Traceback (most recent call last):
  File "C:\Users\shree\.gemini\antigravity\brain\de6a02dc-f256-4315-b02a-1b3db67b01ed\scratch\run_notebook_11.py", line 69, in <module>
    exec(source_code, global_scope)
  File "<string>", line 2, in <module>
  File "d:\Infosys springboards\infrascan\ml\segmentation\evaluate_seg_test.py", line 792, in check_crack500_leakage
    total_no_near_dup = len(test_df) - total_near_dup
                            ^^^^^^^
NameError: name 'test_df' is not defined


### 6. Single-Pass Cross-Domain Generalization Test (CFD Dataset)
Evaluate the external CFD benchmark (118 images) to test cross-domain generalization under unseen conditions.

In [ ]:
# Single-pass evaluation on external CFD test set
cfd_manifest = MANIFESTS_DIR / "cfd.csv"

cfd_metrics, cfd_per_image_df = est.evaluate_dataset_split(
    model=model,
    split_name="cfd",
    manifest_csv=cfd_manifest,
    threshold=winner_info["optimal_threshold"],
    eval_dir=EVAL_DIR,
    device=DEVICE,
    model_name=winner_info["model_name"],
    force_rerun=FORCE_RERUN,
    max_prob_maps=50
)

print("\n--- CFD External Test Set Metrics Summary ---")
for k, v in cfd_metrics.items():
    print(f"  {k}: {v}")


### 7. Inference Latency, Model Complexity & SLA Benchmark
Measure model size in MB, count parameters, and profile inference latency over 10 warmup and 100 timed runs.

In [ ]:
# Profile model size and inference latency
speed_results = est.benchmark_speed_and_size(
    model=model,
    ckpt_path=winner_info["checkpoint_path"],
    device=DEVICE,
    input_size=(384, 384),
    warmup=10,
    runs=100
)

# Save benchmark results
speed_json_path = EVAL_DIR / "speed_and_size.json"
with open(speed_json_path, "w", encoding="utf-8") as f:
    json.dump(speed_results, f, indent=2)

print("=" * 80)
print("[SPEED & COMPLEXITY BENCHMARK]")
print(f"  Model File Size:       {speed_results['model_file_size_mb']} MB")
print(f"  Total Parameters:      {speed_results['parameter_count_total']:,}")
print(f"  Benchmark Device:      {speed_results['benchmark_device']}")
print(f"  Mean Latency (1x384):  {speed_results['mean_latency_ms']:.2f} ms")
print(f"  Median Latency:        {speed_results['median_latency_ms']:.2f} ms")
print(f"  95th Percentile:       {speed_results['p95_latency_ms']:.2f} ms")
print(f"  Throughput:            {speed_results['frames_per_second']:.2f} FPS")
print(f"  Hardware Note: {speed_results['hardware_sla_note']}")
print("=" * 80)


### 8. Generate Summary Report (summary.md) & Print Final Comparison Table
Compile comprehensive `summary.md` and display final comparative table contrasting validation, in-domain test, and cross-domain test metrics.

In [ ]:
# Load thickness results DataFrames
c500_thick_csv = EVAL_DIR / "crack500" / "thickness_results.csv"
cfd_thick_csv = EVAL_DIR / "cfd" / "thickness_results.csv"

c500_thick_df = pd.read_csv(c500_thick_csv) if c500_thick_csv.exists() else pd.DataFrame()
cfd_thick_df = pd.read_csv(cfd_thick_csv) if cfd_thick_csv.exists() else pd.DataFrame()

# Generate summary.md
summary_md_path = EVAL_DIR / "summary.md"
est.generate_summary_markdown(
    winner_info=winner_info,
    val_metrics=val_results,
    c500_metrics=c500_metrics,
    cfd_metrics=cfd_metrics,
    c500_thickness=c500_thick_df,
    cfd_thickness=cfd_thick_df,
    leakage_report=leakage_report,
    speed_results=speed_results,
    output_path=summary_md_path
)

print(f"[OK] Summary report generated at: {summary_md_path}")

# Print final comparison table
summary_table_data = [
    {
        "Split / Dataset": "Crack500 Validation",
        "Threshold": f"{val_results['threshold']:.2f}",
        "Per-Image Dice": f"{val_results['val_dice']:.4f}",
        "95% Bootstrap CI (Dice)": "—",
        "Per-Image IoU": f"{val_results['val_iou']:.4f}",
        "95% Bootstrap CI (IoU)": "—",
        "Precision": f"{val_results['val_precision']:.4f}",
        "Recall": f"{val_results['val_recall']:.4f}",
        "Boundary F1": "—",
        "SLA Status": "Reference"
    },
    {
        "Split / Dataset": "Crack500 Test (In-Domain)",
        "Threshold": f"{c500_metrics['operating_threshold']:.2f}",
        "Per-Image Dice": f"{c500_metrics['mean_dice_per_image']:.4f}",
        "95% Bootstrap CI (Dice)": f"[{c500_metrics['bootstrap_dice_95_ci'][0]:.4f}, {c500_metrics['bootstrap_dice_95_ci'][1]:.4f}]",
        "Per-Image IoU": f"{c500_metrics['mean_iou_per_image']:.4f}",
        "95% Bootstrap CI (IoU)": f"[{c500_metrics['bootstrap_iou_95_ci'][0]:.4f}, {c500_metrics['bootstrap_iou_95_ci'][1]:.4f}]",
        "Precision": f"{c500_metrics['mean_precision_per_image']:.4f}",
        "Recall": f"{c500_metrics['mean_recall_per_image']:.4f}",
        "Boundary F1": f"{c500_metrics['mean_boundary_f1_per_image']:.4f}",
        "SLA Status": "Below Target (0.85/0.75)" if c500_metrics['mean_dice_per_image'] < 0.85 else "Target Met"
    },
    {
        "Split / Dataset": "CFD Test (Cross-Domain)",
        "Threshold": f"{cfd_metrics['operating_threshold']:.2f}",
        "Per-Image Dice": f"{cfd_metrics['mean_dice_per_image']:.4f}",
        "95% Bootstrap CI (Dice)": f"[{cfd_metrics['bootstrap_dice_95_ci'][0]:.4f}, {cfd_metrics['bootstrap_dice_95_ci'][1]:.4f}]",
        "Per-Image IoU": f"{cfd_metrics['mean_iou_per_image']:.4f}",
        "95% Bootstrap CI (IoU)": f"[{cfd_metrics['bootstrap_iou_95_ci'][0]:.4f}, {cfd_metrics['bootstrap_iou_95_ci'][1]:.4f}]",
        "Precision": f"{cfd_metrics['mean_precision_per_image']:.4f}",
        "Recall": f"{cfd_metrics['mean_recall_per_image']:.4f}",
        "Boundary F1": f"{cfd_metrics['mean_boundary_f1_per_image']:.4f}",
        "SLA Status": "Below Target (0.85/0.75)" if cfd_metrics['mean_dice_per_image'] < 0.85 else "Target Met"
    }
]

df_summary = pd.DataFrame(summary_table_data)
print("\n" + "=" * 115)
print("FINAL SEGMENTATION EVALUATION SUMMARY TABLE")
print("=" * 115)
print(df_summary.to_string(index=False))
print("=" * 115)


### 9. Deliverables Inventory
Enumerate all generated masks, probability maps, example panels, metrics files, and reports.

In [ ]:
# Enumerate all created evaluation artifacts
print("=" * 80)
print("INFRASCAN SEGMENTATION EVALUATION DELIVERABLES INVENTORY")
print(f"Base Directory: {EVAL_DIR}")
print("=" * 80)

def count_files(p: Path, ext: str = "*.png") -> int:
    return len(list(p.glob(ext))) if p.exists() else 0

print(f"\n1. Crack500 Deliverables:")
print(f"   - Predicted Masks (0/255 PNG):     {count_files(EVAL_DIR / 'crack500' / 'masks')} files")
print(f"   - Probability Maps (8-bit PNG):     {count_files(EVAL_DIR / 'crack500' / 'probs')} files")
print(f"   - Qualitative Comparison Panels:    {count_files(EVAL_DIR / 'crack500' / 'example_panels')} files")
print(f"   - Per-Image Metrics CSV:           {(EVAL_DIR / 'crack500' / 'per_image_metrics.csv').exists()}")
print(f"   - Thickness Breakdown CSV:         {(EVAL_DIR / 'crack500' / 'thickness_results.csv').exists()}")
print(f"   - Thickness Bar Chart PNG:         {(EVAL_DIR / 'crack500' / 'thickness_results.png').exists()}")
print(f"   - Test Metrics JSON:               {(EVAL_DIR / 'crack500_test_metrics.json').exists()}")

print(f"\n2. CFD External Deliverables:")
print(f"   - Predicted Masks (0/255 PNG):     {count_files(EVAL_DIR / 'cfd' / 'masks')} files")
print(f"   - Probability Maps (8-bit PNG):     {count_files(EVAL_DIR / 'cfd' / 'probs')} files")
print(f"   - Qualitative Comparison Panels:    {count_files(EVAL_DIR / 'cfd' / 'example_panels')} files")
print(f"   - Per-Image Metrics CSV:           {(EVAL_DIR / 'cfd' / 'per_image_metrics.csv').exists()}")
print(f"   - Thickness Breakdown CSV:         {(EVAL_DIR / 'cfd' / 'thickness_results.csv').exists()}")
print(f"   - Thickness Bar Chart PNG:         {(EVAL_DIR / 'cfd' / 'thickness_results.png').exists()}")
print(f"   - Test Metrics JSON:               {(EVAL_DIR / 'cfd_test_metrics.json').exists()}")

print(f"\n3. Governance & System Deliverables:")
print(f"   - Leakage Report JSON:             {(EVAL_DIR / 'leakage_report.json').exists()}")
print(f"   - Speed & Size Benchmark JSON:     {(EVAL_DIR / 'speed_and_size.json').exists()}")
print(f"   - Executive Markdown Summary:      {(EVAL_DIR / 'summary.md').exists()}")
print("=" * 80)
